# H-004 · Beta Feature Suite

Factor test for **H-004** (equities): whether beta-derived features (asymmetric betas, Blume adjustment, residual momentum, and Fama–French–style smart-beta loadings) carry cross-sectional predictive power for forward returns at Alphalens `periods=(1, 5, 21)` (primary narrative **5d**).

- **Idea** — Build eleven cross-sectional features from rolling SPY OLS and a 4-factor (Carhart-style) regression; screen a balanced window grid on research IS only.
- **Claim** — Asymmetric betas / residual momentum / smart-beta loadings improve next-week IC beyond raw market beta.
- **Why it might work** — High downside beta is under-compensated for crash risk (Ang, Chen & Xing 2006); residual momentum isolates stock-specific drift after stripping systematic factors (Blitz, Huij & Martens 2011).
- **Data** — Daily OHLCV long panel (`s1_factor_panel_train.parquet`) + SPY daily returns + **ETF Tier A** Carhart proxies via `fetch_ff_factors_daily` (SPY / IWM / IWD / IWF / MTUM / BIL). Same schema as Ken French (`mkt_rf, smb, hml, mom, rf`) so `benchmark='ff'` is unchanged.

## Learning note (why not Ken French ZIPs)

The Ken French Data Library is free and research-grade but **monthly-lagged**, **historically revised**, and therefore **not point-in-time** for live trading or train–serve parity. Training smart-* features on Dartmouth ZIPs and deploying on a different factor construction would be covariate shift.

- **Archived ZIP fetcher:** `02_research/notebooks/s1_equities/redundant/old_fama_french_fetcher.py`
- **Archived notebook (Ken French path):** `02_research/notebooks/s1_equities/redundant/old_H-004_beta.ipynb`
- **Active fetcher:** `data.ingestion.alternative_data.fama_french_fetcher` (ETF proxies)

**Transferable lesson:** freeze only features you can recompute on the decision clock.

## Features (11 families via 8 store callers)

| Store caller | Output column(s) | `normalize` |
|---|---|---|
| `add_beta(benchmark='spy')` | `beta_{W}` | True (CS pct-rank) |
| `add_beta(benchmark='ff')` | `smart_beta_smb/hml/mom_{W}` | True |
| `add_downside_beta` | `downside_beta_{W}` | True |
| `add_upside_beta` | `upside_beta_{W}` | True |
| `add_net_beta_spread` | `net_beta_spread_{W}` | True |
| `add_relative_downside_beta` | `rel_downside_beta_{W}` | True |
| `add_relative_upside_beta` | `rel_upside_beta_{W}` | True |
| `add_blume_beta` | `blume_beta_{W}` | **none** (never CS-ranked) |
| `add_residual_momentum(benchmark='spy')` | `residual_mom_{K}_{S}` | **none** (never CS-ranked) |
| `add_residual_momentum(benchmark='ff')` | `smart_residual_mom_{K}_{S}` | **none** (never CS-ranked) |

**Workspace pattern:** the first SPY / FF store call runs OLS once per window and caches `_ws_*` columns; later callers reuse them. This notebook drops workspace columns before saving / evaluating.

**Normalize policy:** use library defaults for callers that expose `normalize`. Blume beta and residual momentum have no `normalize` kwarg and are never CS-ranked.

## Beta feature parquet cache

| Path | Role |
|------|------|
| `01_data/data_files/s1_equities/s1_factor_panel_train.parquet` | Research IS OHLCV (cold path only) |
| `01_data/data_files/s1_equities/s1_h004_beta_panel.parquet` | **Notebook artifact** — cleaned IS + all H-004 factor columns (no `_ws_*`) |

**Load gate:** if `s1_h004_beta_panel.parquet` exists and `FORCE_REBUILD` is False → load it into `panel` and **skip** §2 cleaning and §3 OLS rebuild.

**Save gate:** on a cold build, after features are built and workspace columns dropped, write the beta parquet **before** any Alphalens screen / tear sheet.

**Invalidate** by deleting the beta parquet or setting `FORCE_REBUILD = True` when changing `WINDOWS` / `FORMATION_WINDOWS` / `SKIPS`, after store-code changes, **or after switching factor backend (Ken French → ETF)** — smart_* columns change meaning.

This beta parquet is **not** a substitute for the train panel in other notebooks and must **not** be used for OOS / `feature_spec` freeze beyond this H-004 screen.

## Research IS discipline

- Cold path loads **`s1_factor_panel_train.parquet` only** — do not re-split.
- Do **not** use `s1_factor_panel_full.parquet` for window keep/kill.
- Overlapping 5d / 21d labels warrant purge/embargo in later walk-forward; this notebook screens IC on research IS only.
- Variant count = number of H-004 factor columns screened (expected **92**).

Use `data.processing.s1_feature_store` callers — do not reimplement OLS inline.


## 0. Imports & Config

Resolve the repo root; configure the balanced window grid, beta-cache paths, and Alphalens periods. Set `FORCE_REBUILD = True` to ignore an existing beta parquet and rebuild from the train IS (required once after the Ken French → ETF factor swap).


In [1]:
import os
import sys
import time

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

from data.ingestion.equity_fetcher import fetch_ohlcv
from data.ingestion.alternative_data.fama_french_fetcher import fetch_ff_factors_daily
from data.processing.cleaner import forward_fill_panel
from data.processing.feature_implementation.beta_features import market_return_frame
from data.processing.feature_implementation.beta_features import parse_beta_factor_name
from data.processing.s1_feature_store import (
    add_beta,
    add_blume_beta,
    add_downside_beta,
    add_net_beta_spread,
    add_relative_downside_beta,
    add_relative_upside_beta,
    add_residual_momentum,
    add_upside_beta,
    drop_beta_workspace,
)

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
BETA_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_h004_beta_panel.parquet"
)
TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "s1_equities", "factor_tests", "tearsheets"
)

# --- Window screen (edit these lists) ---
WINDOWS = [42, 63, 84, 126, 189, 252]          # OLS / beta windows
FORMATION_WINDOWS = [84, 126, 189, 252]        # must be subset of WINDOWS
SKIPS = [10, 21, 42, 63]                        # residual-momentum skips (no extra OLS)

EXPECTED_N_FACTORS = (
    10 * len(WINDOWS) + 2 * len(FORMATION_WINDOWS) * len(SKIPS)
)  # 60 + 32 = 92

# --- Fixed for this notebook ---
FORCE_REBUILD = True   # True after Ken French -> ETF swap / window changes; then set False
BENCHMARK = "SPY"
PERIODS = (1, 5, 21)    # primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35

print(f"ROOT={ROOT}")
print(f"EXPECTED_N_FACTORS={EXPECTED_N_FACTORS}")
print(f"FORCE_REBUILD={FORCE_REBUILD}")


ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
EXPECTED_N_FACTORS=92
FORCE_REBUILD=True


## 1. Data Loading

### Beta parquet cache contract

1. If `FORCE_REBUILD` is False **and** `s1_h004_beta_panel.parquet` exists → **CACHE HIT**: load it into `panel` (features already present). Skip SPY / FF fetch (Alphalens prices use `panel["close"]`).
2. Otherwise → **CACHE MISS / cold build**: load `s1_factor_panel_train.parquet`, then fetch SPY and ETF Tier A Carhart proxies (`fetch_ff_factors_daily`) with a **40-day forward buffer** so Alphalens can form 21d forward returns near the last IS date.
3. On a warm load, validate H-004 column count via `parse_beta_factor_name`. If the cache looks stale (wrong count), fall through to a cold build.

Do **not** apply another 70/30 split here — the train parquet is already research IS.


In [2]:
use_beta_cache = (not FORCE_REBUILD) and os.path.exists(BETA_PANEL_PATH)
market_returns = None
ff_factors = None

if use_beta_cache:
    panel = pd.read_parquet(BETA_PANEL_PATH)
    panel["date"] = pd.to_datetime(panel["date"])
    FACTOR_COLS = [c for c in panel.columns if parse_beta_factor_name(c) is not None]
    if len(FACTOR_COLS) != EXPECTED_N_FACTORS:
        print(
            f"CACHE STALE: found {len(FACTOR_COLS)} H-004 cols "
            f"(expected {EXPECTED_N_FACTORS}) - falling back to cold build"
        )
        use_beta_cache = False
    else:
        print(f"CACHE HIT: loaded {BETA_PANEL_PATH}")
        n_tickers = panel["ticker"].nunique()
        n_dates = panel["date"].nunique()
        print(
            f"rows={len(panel):,}  tickers={n_tickers}  dates={n_dates:,}  "
            f"factor cols={len(FACTOR_COLS)}  "
            f"[{panel['date'].min().date()} -> {panel['date'].max().date()}]"
        )

if not use_beta_cache:
    panel = pd.read_parquet(TRAIN_PANEL_PATH)
    panel = panel.copy()
    panel["date"] = pd.to_datetime(panel["date"])
    required = {"date", "ticker", "close"}
    missing = required - set(panel.columns)
    if missing:
        raise ValueError(f"train panel missing columns: {sorted(missing)}")

    print(f"CACHE MISS: loaded {TRAIN_PANEL_PATH} (will build H-004 features)")
    n_tickers = panel["ticker"].nunique()
    n_dates = panel["date"].nunique()
    print(
        f"rows={len(panel):,}  tickers={n_tickers}  dates={n_dates:,}  "
        f"[{panel['date'].min().date()} -> {panel['date'].max().date()}]"
    )

    start = panel["date"].min().strftime("%Y-%m-%d")
    end = (panel["date"].max() + pd.Timedelta(days=40)).strftime("%Y-%m-%d")
    spy = fetch_ohlcv(BENCHMARK, start, end)
    market_returns = market_return_frame(spy)
    ff_factors = fetch_ff_factors_daily(start, end)
    print(f"SPY market returns: {len(market_returns):,} rows")
    print(f"FF factors: {len(ff_factors):,} rows  cols={list(ff_factors.columns)}")
    FACTOR_COLS = []

panel.head()


CACHE MISS: loaded c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_factor_panel_train.parquet (will build H-004 features)
rows=289,287  tickers=100  dates=2,914  [2010-01-04 -> 2021-07-30]
SPY market returns: 2,941 rows
FF factors: 2,113 rows  cols=['date', 'mkt_rf', 'smb', 'hml', 'mom', 'rf']


,date,ticker,open,high,low,close,volume
0,2010-01-04,AAPL,6.389116,6.421147,6.357684,6.406478,493729600.0
1,2010-01-04,ABT,17.989170,18.111998,17.899539,18.078800,10829095.0
2,2010-01-04,ADBE,36.650002,37.299999,36.650002,37.090000,4710200.0
3,2010-01-04,AET,29.090984,30.016525,28.918580,29.943932,5671979.0
4,2010-01-04,AIG,18.950967,18.957175,18.255746,18.553698,7750900.0


## 2. Data Cleaning & Engineering

**Cold path only:** forward-fill `close` (`limit=5`) then drop residual nulls.

**Warm path:** skip — the beta parquet already holds the cleaned panel + factors. No floor / winsorize in the store API.


In [3]:
if use_beta_cache:
    print("Skipping §2 - panel loaded from beta cache")
else:
    panel = forward_fill_panel(panel, columns=["close"], limit=5)
    panel = panel.dropna(subset=["close"]).reset_index(drop=True)
    print(
        f"after clean: rows={len(panel):,}  "
        f"null close={(panel['close'].isna().sum())}"
    )


after clean: rows=289,287  null close=0


## 3. Modeling / Signal Construction

### 3.1 H-004 beta feature suite

All SPY / FF callers share the same `WINDOWS` list so workspace OLS runs **once per window**. `FORMATION_WINDOWS` must be a subset of `WINDOWS`; `skip` only changes residual aggregation (no extra regression).

**Save gate (cold path):** after build + `drop_beta_workspace`, write `s1_h004_beta_panel.parquet` **before** Alphalens. Warm path skips the rebuild and re-derives `FACTOR_COLS` from the cached panel.

If runtime exceeds ~20 minutes on a cold run, trim `WINDOWS` in §0.


In [4]:
t0 = time.perf_counter()

if use_beta_cache:
    print("Skipping §3 rebuild - using cached H-004 columns")
    FACTOR_COLS = [c for c in panel.columns if parse_beta_factor_name(c) is not None]
else:
    # SPY family (7 single-window features)
    panel = add_beta(panel, market_returns, benchmark="spy", windows=WINDOWS)
    panel = add_downside_beta(panel, market_returns, windows=WINDOWS)
    panel = add_upside_beta(panel, market_returns, windows=WINDOWS)
    panel = add_net_beta_spread(panel, market_returns, windows=WINDOWS)
    panel = add_relative_downside_beta(panel, market_returns, windows=WINDOWS)
    panel = add_relative_upside_beta(panel, market_returns, windows=WINDOWS)
    panel = add_blume_beta(panel, market_returns, windows=WINDOWS)

    # FF family (3 smart betas)
    panel = add_beta(panel, ff_factors, benchmark="ff", windows=WINDOWS)

    # Residual momentum (CAPM + 4-factor)
    panel = add_residual_momentum(
        panel,
        market_returns,
        benchmark="spy",
        formation_window=FORMATION_WINDOWS,
        skip=SKIPS,
    )
    panel = add_residual_momentum(
        panel,
        ff_factors,
        benchmark="ff",
        formation_window=FORMATION_WINDOWS,
        skip=SKIPS,
    )

    panel = drop_beta_workspace(panel)
    FACTOR_COLS = [c for c in panel.columns if parse_beta_factor_name(c) is not None]
    assert len(FACTOR_COLS) == EXPECTED_N_FACTORS, (
        f"expected {EXPECTED_N_FACTORS} factor cols, got {len(FACTOR_COLS)}"
    )

    # Persist BEFORE Alphalens (cold path)
    os.makedirs(os.path.dirname(BETA_PANEL_PATH), exist_ok=True)
    panel.to_parquet(BETA_PANEL_PATH, index=False)
    print(f"Wrote beta feature panel -> {BETA_PANEL_PATH}")

elapsed = time.perf_counter() - t0
print(f"H-004 factor columns: {len(FACTOR_COLS)}  (build/load wall={elapsed:.1f}s)")
assert len(FACTOR_COLS) == EXPECTED_N_FACTORS, (
    f"expected {EXPECTED_N_FACTORS} factor cols, got {len(FACTOR_COLS)}"
)
assert not any(c.startswith("_ws_") for c in panel.columns), "_ws_* columns still present"
print("Factor columns:")
for c in sorted(FACTOR_COLS):
    print(f"  {c}")


Wrote beta feature panel -> c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h004_beta_panel.parquet
H-004 factor columns: 92  (build/load wall=623.7s)
Factor columns:
  beta_126
  beta_189
  beta_252
  beta_42
  beta_63
  beta_84
  blume_beta_126
  blume_beta_189
  blume_beta_252
  blume_beta_42
  blume_beta_63
  blume_beta_84
  downside_beta_126
  downside_beta_189
  downside_beta_252
  downside_beta_42
  downside_beta_63
  downside_beta_84
  net_beta_spread_126
  net_beta_spread_189
  net_beta_spread_252
  net_beta_spread_42
  net_beta_spread_63
  net_beta_spread_84
  rel_downside_beta_126
  rel_downside_beta_189
  rel_downside_beta_252
  rel_downside_beta_42
  rel_downside_beta_63
  rel_downside_beta_84
  rel_upside_beta_126
  rel_upside_beta_189
  rel_upside_beta_252
  rel_upside_beta_42
  rel_upside_beta_63
  rel_upside_beta_84
  residual_mom_126_10
  residual_mom_126_21
  residual_mom_126_42
  residual_mom_126_63
  res

## 4. Evaluation

Alphalens runs only after `panel` is loaded from the beta parquet **or** freshly written to it — never mid-build.

Screen every H-004 column at `periods=(1, 5, 21)` with `quantiles=5`. Primary sort key: **`ic_5d`**. Decode column parameters with `parse_beta_factor_name`.


In [5]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide close matrix for Alphalens only (dates x tickers)."""
    prices = panel.pivot(index="date", columns="ticker", values="close")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', ...) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5-Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    Alphalens calls plt.show() after each plot, which clears figures under Agg.
    Temporarily replace plt.show so each figure is written into the PDF before close.
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel - pick a screened column "
            f"(available: {FACTOR_COLS})"
        )
    plt.close("all")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-004_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


### 4.1 Window screen summary

Full IC / spread table sorted by `ic_5d`, plus a **best-by-family** table (one row per feature stem).

**Expected signs (literature / intuition):**
- `beta` / smart betas — context-dependent
- `downside_beta`, `rel_downside_beta` — often positive IC for high downside beta (Ang et al.)
- `residual_mom`, `smart_residual_mom` — positive IC for high residual momentum
- Compare each family's best window vs the `beta` baseline at a similar `W` where possible


In [6]:
t0 = time.perf_counter()
prices = to_alphalens_prices(panel)
rows = []
for col in FACTOR_COLS:
    meta = parse_beta_factor_name(col) or {}
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append({"factor": col, **meta, **metrics})

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)
print(f"Alphalens screen wall={time.perf_counter() - t0:.1f}s  n={len(summary)}")

summary["feature_family"] = summary["factor"].map(
    lambda c: (parse_beta_factor_name(c) or {}).get("feature")
)
best_by_family = (
    summary.sort_values("ic_5d", ascending=False)
    .groupby("feature_family", as_index=False)
    .first()
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)

print("\n=== Best by feature family (ic_5d) ===")
print(best_by_family.to_string())

print("\n=== Full screen (sorted by ic_5d) ===")

with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(summary)

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Alphalens screen wall=1065.3s  n=92

=== Best by feature family (ic_5d) ===
        feature_family                     factor             feature  window     ic_1d  spread_1d     ic_5d     spread_5d    ic_21d  spread_21d      K     S
0   smart_residual_mom  smart_residual_mom_189_42  smart_residual_mom     NaN  0.009161   0.000214  0.015781  1.057474e-03  0.036117    0.004279  189.0  42.0
1    rel_downside_beta      rel_downside_beta_252   rel_downside_beta   252.0  0.006535   0.000428  0.013537  1.851255e-03  0.032943    0.008049    NaN   NaN
2       smart_beta_mom         smart_beta_mom_126      smart_beta_mom   126.0  0.010606   0.000052  0.009720 -7.630758e-07 -0.005522   -0.002647    NaN   NaN
3       smart_beta_smb          smart_beta_smb_84      smart_beta_smb    84.0  0.001808   0.000417  0.009412  2.198209e-03  0.016292    0.007738    NaN   NaN
4      rel_upside_beta        rel_upside_beta_126     rel_upside_beta   126.0  0.007242   0.000174  0.009350  6.011669e-04  0.008629  

,factor,feature,window,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d,K,S,feature_family
0,smart_residual_mom_189_42,smart_residual_mom,NaN,0.009161,2.140574e-04,0.015781,1.057474e-03,0.036117,0.004279,189.0,42.0,smart_residual_mom
1,smart_residual_mom_189_63,smart_residual_mom,NaN,0.009438,3.505460e-04,0.015555,1.875600e-03,0.029069,0.006454,189.0,63.0,smart_residual_mom
2,rel_downside_beta_252,rel_downside_beta,252.0,0.006535,4.282595e-04,0.013537,1.851255e-03,0.032943,0.008049,NaN,NaN,rel_downside_beta
3,smart_residual_mom_189_21,smart_residual_mom,NaN,0.008266,2.228039e-04,0.011449,1.426738e-03,0.033263,0.005292,189.0,21.0,smart_residual_mom
4,rel_downside_beta_189,rel_downside_beta,189.0,0.005069,2.728261e-04,0.010924,1.659669e-03,0.029477,0.007660,NaN,NaN,rel_downside_beta
5,smart_residual_mom_126_42,smart_residual_mom,NaN,0.006281,1.470190e-04,0.009834,1.144957e-03,0.025551,0.006346,126.0,42.0,smart_residual_mom
6,smart_beta_mom_126,smart_beta_mom,126.0,0.010606,5.232996e-05,0.009720,-7.630758e-07,-0.005522,-0.002647,NaN,NaN,smart_beta_mom
7,smart_beta_smb_84,smart_beta_smb,84.0,0.001808,4.167652e-04,0.009412,2.198209e-03,0.016292,0.007738,NaN,NaN,smart_beta_smb
8,rel_upside_beta_126,rel_upside_beta,126.0,0.007242,1.736734e-04,0.009350,6.011669e-04,0.008629,0.001086,NaN,NaN,rel_upside_beta
9,net_beta_spread_63,net_beta_spread,63.0,0.005852,1.818698e-04,0.009275,3.179777e-04,0.009016,-0.000546,NaN,NaN,net_beta_spread


### 4.2 Full tear sheets

Edit `TEAR_FACTORS` after reviewing §4.1. Each tear is displayed in-notebook **and** saved as a multi-page PDF under:

`02_research/notebooks/s1_equities/factor_tests/tearsheets/H-004_{factor_col}.pdf`

Filename encodes the factor stem and window arguments (e.g. `H-004_residual_mom_126_21.pdf`). Re-running overwrites the same paths. **Do not** tear all 92 columns (runtime budget).


In [8]:
# Edit after reviewing §4.1 (defaults are placeholders from the plan)
TEAR_FACTORS = [
    "smart_residual_mom_189_42",
    "rel_downside_beta_252",
    "smart_beta_mom_126",
    "smart_beta_smb_84",
    "rel_upside_beta_126",
    "net_beta_spread_63",
    "downside_beta_189",
    "residual_mom_252_21",
    "upside_beta_126",
    "blume_beta_189",
    "beta_189",
    "smart_beta_hml_252",
    "smart_beta_hml_42"
]

for tear_col in TEAR_FACTORS:
    print(f"\n===== Tear sheet: {tear_col} =====")
    run_full_tear(panel, tear_col, prices)



===== Tear sheet: smart_residual_mom_189_42 =====


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1,-0.280489,-0.031971,-0.100088,0.032852,33780,20.245851
2,-0.095874,0.014460,-0.041192,0.016424,33130,19.856277
3,-0.045074,0.051938,-0.002045,0.015006,33030,19.796343
4,-0.010923,0.093036,0.036071,0.016809,33130,19.856277
5,0.022746,0.335100,0.098595,0.038080,33779,20.245252


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.072,0.181,0.395
Quantile 2 Mean Turnover,0.176,0.399,0.649
Quantile 3 Mean Turnover,0.215,0.454,0.683
Quantile 4 Mean Turnover,0.191,0.415,0.651
Quantile 5 Mean Turnover,0.081,0.201,0.418


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.979,0.906,0.666


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-004_smart_beta_hml_42.pdf (3 pages)


## 5. Conclusion

- **Variants tried:** `EXPECTED_N_FACTORS` (= 92 with the locked balanced grid) H-004 factor columns on research IS only.
- **Primary metric:** `ic_5d` (also report 1d / 21d). Use §4.1 `best_by_family` to shortlist 1–2 combos per surviving family for a later `feature_spec` freeze — do **not** integrate into `s1_factor_panel.ipynb` yet.
- **Cache:** this run was a CACHE HIT or COLD build depending on §1; invalidate `s1_h004_beta_panel.parquet` (or set `FORCE_REBUILD = True`) when changing windows, store code, **or the factor backend** (Ken French ZIP → ETF Tier A).
- **Factor source:** ETF proxies via `data.ingestion.alternative_data.fama_french_fetcher` (archived Ken French path under `02_research/notebooks/s1_equities/redundant/`).
- **Holdout:** reserved; do not peek at `s1_factor_panel_full.parquet` for keep/kill.
- **Tear PDFs:** three files under `02_research/notebooks/s1_equities/factor_tests/tearsheets/` named `H-004_{factor_col}.pdf`.
- If cold-run wall time exceeded ~20 minutes, OLS dominated — trim `WINDOWS` in §0 and rebuild.
